# Document Conversion with Docling


## Lab 1: Document Conversion

This is the first of seven labs. They go from converting documents with Docling to building RAG systems, running Docling as a service, and giving AI agents Docling tools.

## What you'll learn

AI applications need documents in a form a model can use. In this lab you'll convert PDFs and other files with Docling and keep their structure (headings, tables, figures, and reading order), which the later labs depend on.

By the end of this lab, you'll know how to:
- Convert PDFs, Word documents, PowerPoint files, and more
- Explore the structure Docling extracts
- Export tables to pandas, CSV, and HTML
- Extract page images and figures
- Describe pictures with a vision model
- Convert a document with the Granite-Docling vision-language model

---

## Introduction

[Docling](https://docling-project.github.io/docling/) is an open-source toolkit that parses documents and converts them into formats generative AI applications can use. It:
- Reads PDF, DOCX, PPTX, XLSX, HTML, images, and more
- Understands PDF page layout, reading order, tables, code blocks, and formulas
- Represents every format with the same `DoclingDocument` structure
- Exports to Markdown, HTML, JSON, and DocTags
- Runs locally, so sensitive documents don't have to leave your machine
- Integrates with LangChain, LlamaIndex, and other AI frameworks

## Why conversion matters

LLMs work with text, but much of the information an organization has sits in PDFs, slides, and spreadsheets. Simple text extraction loses a lot of it: the reading order gets jumbled, tables turn into a stream of numbers, figures disappear, and headings become ordinary lines. Docling keeps that structure, so tables stay tables, figures can be extracted and described, and each item keeps its place in the document's hierarchy.

## Installation and Setup

### Setting up the environment

Ensure you are running Python 3.11 or later in a freshly created virtual environment.

In [ ]:
import sys
assert sys.version_info >= (3, 11) and sys.version_info < (3, 14), "Use Python 3.11, 3.12, or 3.13 to run this notebook."

### Basic Installation

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "docling[vlm]" docling matplotlib pillow pandas ipywidgets python-dotenv \
    "git+https://github.com/ibm-granite-community/utils.git"
! echo "::endgroup::"

### Import Essential Components




In [ ]:
from pathlib import Path

# Core Docling imports
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import PdfFormatOption

# For advanced features
from docling_core.types.doc import ImageRefMode, PictureItem, TableItem, TextItem, DoclingDocument

# For data processing and visualization
import matplotlib.pyplot as plt

# Create output directory
output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

### Optional: Run Docling on a hosted service

Docling runs on this machine (or Colab runtime) by default. You can instead send the conversions to a Docling service: [Docling for IBM watsonx](https://www.ibm.com/products/docling), IBM's managed Docling service, or any docling-serve instance, such as the one you start in the Docling as a Service lab. The service returns the same `ConversionResult` as a local conversion, so the rest of the notebook works unchanged.

To use a service, set these as environment variables, in a `.env` file, or as Colab secrets before running the next cell:

- `DOCLING_SERVICE_URL`: the service endpoint URL
- `DOCLING_SERVICE_API_KEY`: your API key (leave it unset for a docling-serve that does not require one)

If `DOCLING_SERVICE_URL` is not set, Docling runs locally.

In [ ]:
from ibm_granite_community.notebook_utils import get_env_var

# Each setting comes from the environment, a .env file, or Colab secrets, and is empty if unset
DOCLING_SERVICE_URL = get_env_var("DOCLING_SERVICE_URL", "")
DOCLING_SERVICE_API_KEY = get_env_var("DOCLING_SERVICE_API_KEY", "")
USE_DOCLING_SERVICE = bool(DOCLING_SERVICE_URL)

if USE_DOCLING_SERVICE:
    from docling.datamodel.service.options import ConvertDocumentsOptions
    from docling.service_client import DoclingServiceClient

    print(f"Docling conversions will run on the service at {DOCLING_SERVICE_URL}")
else:
    print("Docling conversions will run locally")

## Basic Document Conversion

### Minimal Example

The simplest way to convert a document is a `DocumentConverter` with its default settings:

### Your turn: Try your own document

The cell below uses the Docling technical report as the input. **Try changing the URL** to another PDF, such as a paper you've been meaning to read or a product datasheet, and re-run this cell and the next two to see Docling convert it.

<details>
<summary>What should I try?</summary>

Some URLs to try:
- `"https://arxiv.org/pdf/2408.09869"` — the original Docling paper (lots of figures)
- `"https://arxiv.org/pdf/2502.09927"` — a paper dense with tables
- Any PDF URL you have on hand

You can also point at a **local file**: `docling_paper = "path/to/my_document.pdf"`. Docling also handles `.docx`, `.html`, `.pptx`, and other formats.

</details>

In [ ]:
# TODO: try changing the URL below to a different PDF or a local file path.
# example document: Docling Technical Report
docling_paper = "https://arxiv.org/pdf/2501.17887"  # <-- change me

In [ ]:
# Simple conversion

# Create a converter instance: local Docling, or the Docling service if you configured one.
# Both return the same ConversionResult.
if USE_DOCLING_SERVICE:
    converter = DoclingServiceClient(url=DOCLING_SERVICE_URL, api_key=DOCLING_SERVICE_API_KEY)
else:
    converter = DocumentConverter()

# Convert a document
result = converter.convert(docling_paper)
doc = result.document

In [ ]:
# Export to Markdown
md_out = doc.export_to_markdown()

# printing an excerpt
print(f"{md_out[:2000]}...")

### Document Structure Exploration

Docling recovers the document's structure: titles, section headings, paragraphs, lists, tables, and pictures, in reading order. Lab 2 (chunking) and Lab 3 (visual grounding) both build on it.

Print some basic counts and the first few items:

In [ ]:
# Document metadata - important for tracking
print(f"Document title: {doc.name}")
print(f"Number of pages: {len(doc.pages)}")
print(f"Number of tables: {len(doc.tables)}")
print(f"Number of pictures: {len(doc.pictures)}")

# Explore the document structure
print("\nDocument structure:")
for i, (item, level) in enumerate(doc.iterate_items()):
    if i < 10:  # Show first 10 items
        item_type = type(item).__name__
        text_preview = item.text[:200] if hasattr(item, 'text') else 'N/A'
        print(f"{'  ' * level}{item_type}: {text_preview}")

## Export Formats and Options

Export the document to Markdown, HTML, JSON (as a dictionary), and DocTags, and save it to disk:

In [ ]:
# Export to different formats (various options available, but called with default ones)
markdown_text = doc.export_to_markdown()
html_text = doc.export_to_html()
json_dict = doc.export_to_dict()
doc_tags = doc.export_to_doctags()

# Save different formats (various options available, some shown)
doc.save_as_markdown(
    output_dir / "document.md",
    image_mode=ImageRefMode.PLACEHOLDER,
    image_placeholder="<!-- my image placeholder -->",
    # ...
)

# ...

# Export as JSON
doc.save_as_json(
    output_dir / "document.json",
)

## Working with Tables

Docling recovers the row and column structure of tables. For this section, we'll use [a paper with more tables](https://arxiv.org/abs/2502.09927).

### Basic Table Export

In [ ]:
# example document with tables
tables_example = "https://arxiv.org/pdf/2502.09927"

In [ ]:
# Convert a document with tables
table_result = converter.convert(tables_example)
table_doc = table_result.document

In [ ]:
print(f"\nDocument contains {len(table_doc.tables)} tables")

# Export all tables
for table_idx, table in enumerate(table_doc.tables):
    # Convert to pandas DataFrame
    df = table.export_to_dataframe(doc=table_doc)

    print(f"\n## Table {table_idx}")
    print(f"Shape: {df.shape}")
    display(df.head())

    # Save as CSV
    df.to_csv(output_dir / f"table_{table_idx}.csv", index=False)

    # Save as HTML
    with open(output_dir / f"table_{table_idx}.html", "w") as fp:
        fp.write(table.export_to_html(doc=table_doc))

## Image and Figure Extraction

Docling can also save page images and crop out individual figures and tables.

### Extracting and Visualizing Page Images

### Your turn: Image resolution

The cell below extracts page images and figures. `IMAGE_RESOLUTION_SCALE` controls how sharp those extracted images are. **Try a different value** and inspect the saved images in `output/images/` to compare.

<details>
<summary>What should I try?</summary>

`IMAGE_RESOLUTION_SCALE` multiplies the base 72 DPI. Try:
- `1.0` — base (72 DPI), smallest files, fuzziest images
- `2.0` — the default (144 DPI), balanced
- `4.0` — sharp (288 DPI), larger files, slower to generate

Higher scales give sharper images, which helps vision models, but use more memory and take longer.

</details>

In [ ]:
# TODO: try changing IMAGE_RESOLUTION_SCALE below to see how extracted image quality differs.
# Configure pipeline to extract images
IMAGE_RESOLUTION_SCALE = 2.0  # <-- change me (2.0 = 144 DPI)

if USE_DOCLING_SERVICE:
    converter_with_images = DoclingServiceClient(
        url=DOCLING_SERVICE_URL,
        api_key=DOCLING_SERVICE_API_KEY,
        options=ConvertDocumentsOptions(
            images_scale=IMAGE_RESOLUTION_SCALE,
            include_page_images=True,
            include_images=True,
            image_export_mode="embedded",  # return the images inside the document
        ),
    )
else:
    pipeline_options = PdfPipelineOptions(
        images_scale=IMAGE_RESOLUTION_SCALE,
        generate_page_images=True,
        generate_picture_images=True,
    )

    converter_with_images = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )

# Convert with image extraction
img_result = converter_with_images.convert(docling_paper)
img_doc = img_result.document

In [ ]:
# Create images subdirectory
images_dir = output_dir / "images"
images_dir.mkdir(exist_ok=True)

# Save page images
for page_no, page in img_doc.pages.items():
    page_image_filename = images_dir / f"page_{page_no}.png"
    with page_image_filename.open("wb") as fp:
        page.image.pil_image.save(fp, format="PNG")

print(f"Saved {len(img_doc.pages)} page images")

### Extract Figures and Tables as Images

Save each figure and table as its own image:

In [ ]:
# Extract all figures and tables as separate images
table_counter = 0
picture_counter = 0

for element, level in img_doc.iterate_items():
    if isinstance(element, TableItem):
        table_counter += 1
        image_filename = images_dir / f"table_{table_counter}.png"
        with image_filename.open("wb") as fp:
            element.get_image(img_doc).save(fp, "PNG")

    elif isinstance(element, PictureItem):
        picture_counter += 1
        image_filename = images_dir / f"figure_{picture_counter}.png"
        with image_filename.open("wb") as fp:
            element.get_image(img_doc).save(fp, "PNG")

print(f"Extracted {table_counter} tables and {picture_counter} figures as images")

### Inspecting Picture Content

For each picture, Docling records its caption from the document and any text found inside it. Display each picture with that information:

In [ ]:
def inspect_pictures_with_images(doc: DoclingDocument, image_size=(6, 4)):
    """Display pictures inline with their text content"""
    for idx, picture in enumerate(doc.pictures):
        print(f"\n{'='*60}")
        print(f"Picture {idx}")
        print(f"{'='*60}")

        # Display the image
        try:
            img = picture.get_image(doc)
            if img:
                plt.figure(figsize=image_size)
                plt.imshow(img)
                plt.axis('off')
                plt.title(f"Picture {idx}")
                plt.show()
        except Exception as e:
            print(f"Could not display image: {e}")

        # Display metadata
        caption = picture.caption_text(doc)
        if caption:
            print(f"\nCaption: {caption}")

        if hasattr(picture, 'prov') and picture.prov:
            print(f"Location: Page {picture.prov[0].page_no}")

        # Display embedded text
        print("\nEmbedded text elements:")
        text_found = False
        for item, level in doc.iterate_items(root=picture, traverse_pictures=True):
            if isinstance(item, TextItem):
                print(f"{'  ' * (level + 1)}- {item.text}")
                text_found = True

        if not text_found:
            print("  (No text elements found)")

# Use the simple inline display
inspect_pictures_with_images(img_doc)

### Visualizing Document Layout with Bounding Boxes

To see how Docling read each page, draw the detected elements on the page images with Docling's built-in layout visualizer:

In [ ]:
from docling_core.transforms.visualizer.layout_visualizer import LayoutVisualizer

layout_visualizer = LayoutVisualizer()
page_images = layout_visualizer.get_visualization(doc=img_doc)

num_pages_to_viz = 2  # first N pages to visualize
pages_to_viz = list(page_images.keys())[:num_pages_to_viz]
for page in pages_to_viz:
    display(page_images[page])

## Enrichment: Describing Pictures

You can run the vision-language model (VLM) that describes pictures in three ways:

1. Docling runs the model locally
1. A local inference server such as Ollama or LM Studio runs it
1. The Docling service runs it, if you configured one above

The next cell picks one based on your environment.

In [ ]:
import urllib.request
import urllib.error

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def is_ollama_available(url="http://localhost:11434"):
    """Check if Ollama is running and accessible."""
    try:
        req = urllib.request.Request(url, method="GET")
        with urllib.request.urlopen(req, timeout=2) as response:
            return response.status == 200
    except (urllib.error.URLError, TimeoutError):
        return False

OLLAMA_AVAILABLE = is_ollama_available()

# Prefer the Docling service if configured, then Ollama (better memory management), then a local VLM
RUN_LOCAL_OLLAMA = not USE_DOCLING_SERVICE and not IN_COLAB and OLLAMA_AVAILABLE
RUN_LOCAL_VLM = not USE_DOCLING_SERVICE and not IN_COLAB and not OLLAMA_AVAILABLE

print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Ollama available: {OLLAMA_AVAILABLE}")
print(f"Using: {'Docling service' if USE_DOCLING_SERVICE else 'Ollama' if RUN_LOCAL_OLLAMA else 'Local VLM' if RUN_LOCAL_VLM else 'Remote'}")

### Picture Description

A multimodal model can write a description of each picture, which is far more useful for search and question answering than the caption alone.

First, define a function that displays the first few pictures with their captions and descriptions:

In [ ]:
from IPython.display import HTML

def display_enriched_document(enr_doc: DoclingDocument, model_name: str = "VLM"):
    html_buffer = []
    # display the first 5 pictures and their captions and annotations:
    for pic in enr_doc.pictures[:5]:
        html_item = (
            f"<h3>Picture <code>{pic.self_ref}</code></h3>"
            f'<img src="{pic.image.uri!s}" /><br />'
            f"<h4>Caption</h4>{pic.caption_text(doc=enr_doc)}<br />"
        )
        # Get VLM description from pic.meta.description
        if pic.meta and pic.meta.description:
            # Use provided model_name, or created_by if it's not "not-implemented"
            source = model_name if pic.meta.description.created_by == "not-implemented" else pic.meta.description.created_by
            html_item += (
                f"<h4>Description ({source})</h4>"
                f"{pic.meta.description.text}<br />\n"
            )
        html_buffer.append(html_item)
    display(HTML("<hr />".join(html_buffer)))

The local path runs Granite Vision through Docling.

### Your turn: Prompt the vision model

The cell below uses a vision-language model (Granite Vision) to **describe each picture in the document**. The `prompt` tells the model what to look for. **Try changing it** to see how much the descriptions change.

<details>
<summary>What should I try?</summary>

Some prompts to try:
- `"Describe this image in one sentence."` — forces a concise description
- `"Extract any text you can read in this image."` — OCR-style
- `"What type of chart is this, and what does it show?"` — targeted at figures and charts
- `"List the main objects and their relationships."` — structural

The model follows the prompt closely, so small changes in wording show up in the output.

**Note:** this exercise works for both the local VLM (next cell) and Ollama (two cells down). If you're on the Ollama path, change the `prompt` in both cells to stay consistent. The Docling service path (three cells down) uses the prompt of the service's picture description model instead.

</details>

In [ ]:
from docling.datamodel.pipeline_options import PictureDescriptionVlmOptions

# TODO: try changing the `prompt` below to ask a different question about each image.
# Configure enrichment pipeline
if RUN_LOCAL_VLM:
    # Configure enrichment pipeline
    enrichment_options = PdfPipelineOptions(
        do_picture_description=True,
        picture_description_options=PictureDescriptionVlmOptions(  # model & prompt choice
            repo_id="ibm-granite/granite-vision-3.3-2b",
            prompt="Give a detailed description of what is depicted in the image",  # <-- change me
        ),
        generate_picture_images=True,
        images_scale=2.0,
    )

    converter_enriched = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=enrichment_options)
        }
    )

    enr_result = converter_enriched.convert(docling_paper)
    enr_doc = enr_result.document

    display_enriched_document(enr_doc, model_name="granite-vision-3.3-2b (Local)")

We can also run it using an OpenAI-compatible API like Ollama.

In [ ]:
from docling.datamodel.pipeline_options import PictureDescriptionApiOptions

if RUN_LOCAL_OLLAMA:
    # Configure enrichment pipeline
    enrichment_options = PdfPipelineOptions(
        do_picture_description=True,
        enable_remote_services = True,
        picture_description_options=PictureDescriptionApiOptions(
            url="http://localhost:11434/v1/chat/completions",
            params={
                "model": "granite3.2-vision:2b",
                "max_completion_tokens": 200,
            },
            prompt="Give a detailed description of what is depicted in the image",
            timeout=60,
        ),
        generate_picture_images=True,
        images_scale=1.0,
    )

    converter_enriched = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=enrichment_options)
        }
    )

    enr_result = converter_enriched.convert(docling_paper)
    enr_doc = enr_result.document

    display_enriched_document(enr_doc, model_name="granite3.2-vision:2b (Ollama)")

With a Docling service configured, the service runs the picture description model for you:

In [ ]:
if USE_DOCLING_SERVICE:
    enrichment_options = ConvertDocumentsOptions(
        do_picture_description=True,
        # picture_description_preset="granite_vision",  # choose a model, if your service offers it
        include_images=True,
        images_scale=2.0,
        image_export_mode="embedded",
    )

    with DoclingServiceClient(
        url=DOCLING_SERVICE_URL,
        api_key=DOCLING_SERVICE_API_KEY,
        options=enrichment_options,
        job_timeout=900,  # describing every picture takes a while
    ) as service:
        enr_doc = service.convert(docling_paper).document

    display_enriched_document(enr_doc, model_name="Docling service")

## Converting with a Vision-Language Model

Instead of its standard pipeline of separate layout, table, and OCR models, Docling can convert each page with a single vision-language model (VLM) trained for document conversion.

### Using Granite-Docling

[Granite-Docling](https://huggingface.co/ibm-granite/granite-docling-258M) is IBM's compact (258M parameters) vision-language model built for document conversion. It outputs [DocTags](https://github.com/docling-project/docling-core), a markup that records hierarchy, tables, and layout, so one model replaces the multi-stage pipeline of layout detection, table structure recognition, and OCR.

The `granite_docling` preset uses the MLX build on Apple Silicon and the Transformers build elsewhere (CPU, CUDA, or XPU).

In [ ]:
from docling.datamodel.pipeline_options import VlmConvertOptions, VlmPipelineOptions
from docling.pipeline.vlm_pipeline import VlmPipeline

if USE_DOCLING_SERVICE:
    # Ask the service to run its VLM pipeline with the same granite_docling preset
    converter_vlm = DoclingServiceClient(
        url=DOCLING_SERVICE_URL,
        api_key=DOCLING_SERVICE_API_KEY,
        options=ConvertDocumentsOptions(pipeline="vlm", vlm_pipeline_preset="granite_docling"),
        job_timeout=900,  # page-by-page VLM conversion takes a while
    )
else:
    # The granite_docling preset picks the inference engine for your hardware:
    # MLX on Apple Silicon, Transformers everywhere else (CPU / CUDA / XPU).
    vlm_pipeline_options = VlmPipelineOptions(
        vlm_options=VlmConvertOptions.from_preset("granite_docling"),
    )

    converter_vlm = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=vlm_pipeline_options,
                pipeline_cls=VlmPipeline,
            )
        }
    )

vlm_result = converter_vlm.convert(docling_paper)
vlm_doc = vlm_result.document

In [ ]:
vlm_md_out = vlm_doc.export_to_markdown()

# printing an excerpt
print(f"{vlm_md_out[:2000]}...")

## Summary

In this lab you:

- Converted a PDF with Docling and explored its structure
- Exported it to Markdown, HTML, JSON, and DocTags
- Exported tables to pandas, CSV, and HTML
- Saved page images, figures, and tables as images, and drew the detected layout
- Described pictures with a vision model
- Converted a document with Granite-Docling

## Next steps

Try converting your own documents and check how well their structure comes through. In **Lab 2** you'll split converted documents into chunks for retrieval, and in **Lab 3** you'll build a RAG system that shows where each answer came from.

For more information:
- GitHub: https://github.com/docling-project/docling
- Documentation: https://docling-project.github.io/docling/
- Technical Report: https://arxiv.org/abs/2408.09869
- Examples: https://github.com/docling-project/docling/tree/main/examples